### DistilBERT Fine-tuning

### Import libraries

In [1]:
!pip install pytorch-lightning transformers torch pandas scikit-learn -q

In [2]:
import torch
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [3]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
train_csv_path = '/content/drive/MyDrive/disaster-tweets/train.csv'
test_csv_path  = '/content/drive/MyDrive/disaster-tweets/test.csv'

In [6]:
train = pd.read_csv(train_csv_path)
test = pd.read_csv(test_csv_path)

print(train.shape)
train.head()

(7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [7]:
train['target'].value_counts()

,count
target,
0,4342
1,3271


### Handle missing values

In [8]:
train['keyword'] = train['keyword'].fillna('none')
test['keyword'] = test['keyword'].fillna('none')

train['location'] = train['location'].fillna('unknown')
test['location'] = test['location'].fillna('unknown')

In [9]:
# Combine keyword + text (no heavy cleaning needed for BERT)
# Note: unlike TF-IDF, we don't remove punctuation/stopwords for BERT -
# BERT understands raw natural text better
train['text_combined'] = train['keyword'] + ' ' + train['text']
test['text_combined'] = test['keyword'] + ' ' + test['text']

In [10]:
# Train/Validation split

train_df, val_df = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train['target']
)

print("Train size:", train_df.shape)
print("Validation size:", val_df.shape)

Train size: (6090, 6)
Validation size: (1523, 6)


### Load pretrained tokenizer

In [11]:
MODEL_NAME = 'distilbert/distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### Data (Dataset class)

In [13]:
class TweetDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True, max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.has_labels = has_labels
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.iloc[idx]['text_combined']

        tokens = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'input_ids': tokens['input_ids'].squeeze(),
            'attention_mask': tokens['attention_mask'].squeeze(),
        }

        if self.has_labels:
            item['labels'] = torch.tensor(self.df.iloc[idx]['target'], dtype=torch.long)

        return item

### Define LightningDataModule

In [14]:
class TweetDataModule(pl.LightningDataModule):
    def __init__(self, train_df, val_df, test_df, tokenizer, batch_size=16):
        super().__init__()
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self.tokenizer = tokenizer
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_ds = TweetDataset(self.train_df, self.tokenizer, has_labels=True)
        self.val_ds = TweetDataset(self.val_df, self.tokenizer, has_labels=True)
        self.test_ds = TweetDataset(self.test_df, self.tokenizer, has_labels=False)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False)

### Create DataModule instance

In [15]:
data_module = TweetDataModule(
    train_df=train_df,
    val_df=val_df,
    test_df=test,
    tokenizer=tokenizer,
    batch_size=16
)
data_module.setup()

### Model

In [16]:
class TweetClassifier(pl.LightningModule):
    def __init__(self, model_name='distilbert/distilbert-base-uncased', num_labels=2, lr=2e-5):
        super().__init__()
        self.save_hyperparameters()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels, ignore_mismatched_sizes=True
        )
        self.lr = lr
        self.validation_step_outputs = []

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
        )
        loss = outputs.loss
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
        )
        loss = outputs.loss
        preds = torch.argmax(outputs.logits, dim=1)

        self.validation_step_outputs.append({'preds': preds, 'labels': batch['labels']})
        self.log('val_loss', loss, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        preds = torch.cat([x['preds'] for x in self.validation_step_outputs]).cpu().numpy()
        labels = torch.cat([x['labels'] for x in self.validation_step_outputs]).cpu().numpy()
        f1 = f1_score(labels, preds)
        self.log('val_f1', f1, prog_bar=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=self.lr)

In [17]:
# Create model instance
model = TweetClassifier(model_name=MODEL_NAME, num_labels=2, lr=2e-5)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Training

In [18]:
trainer = pl.Trainer(
    max_epochs=5,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    log_every_n_steps=20,
)

trainer.fit(model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                                ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ DistilBertForSequenceClassification │ 67.0 M │ eval │     0 │
└───┴───────┴─────────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 67.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 67.0 M                                                                                               
Total estimated model params size (MB): 267.820                                                                    
Modules in train mode: 0                                                                                           
Modules in eval mode: 96                                                                                           
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/loops/fit_loop.py:538: Found 96 module(s) in eval mode at
the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore
this warning.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


### Predict on test set
### Get predictions on test data

In [19]:
model.eval()
model.to(device)

test_loader = data_module.test_dataloader()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

print("Test predictions done. Total:", len(all_preds))

Test predictions done. Total: 3263


### Create submission file

In [23]:
submission = pd.DataFrame({
    'id': test['id'],
    'target': all_preds
})

submission.to_csv('submission_DistilBERT.csv', index=False)
print("submission_DistilBERT.csv created. Shape:", submission.shape)
submission.head(10)

submission_DistilBERT.csv created. Shape: (3263, 2)


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
5,12,1
6,21,0
7,22,0
8,27,0
9,29,0


### Download submission file

In [24]:
from google.colab import files
files.download('submission_DistilBERT.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
from sklearn.metrics import f1_score, classification_report

model.eval()
model.to(device)

val_loader = data_module.val_dataloader()
val_preds = []
val_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        val_preds.extend(preds.cpu().numpy())
        val_labels.extend(labels.numpy())

# Calculate F1 score (same metric Kaggle uses)
f1 = f1_score(val_labels, val_preds)
print("Validation F1 Score:", f1)

# Detailed report
print("\nClassification Report:\n")
print(classification_report(val_labels, val_preds))

Validation F1 Score: 0.8016194331983806

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.90      0.86       869
           1       0.85      0.76      0.80       654

    accuracy                           0.84      1523
   macro avg       0.84      0.83      0.83      1523
weighted avg       0.84      0.84      0.84      1523

